# Magicbricks Localities — Cleaning & Feature Engineering

Reads `magicbricks_localities.xlsx` (600 localities, 16 columns), parses the dense price string,
turns amenity lists into counts, cleans the heavy text blocks, and saves a model-ready table.

This is an **improved, reviewed** version of an initial cleaning script (review below).

## Code review of the original script

**What it did well:** standardised `N/A`/blank to `NaN`, parsed Residential buy/rent via regex,
counted comma-lists, normalised whitespace.

**Issues found & fixed here**

| # | Issue in original | Fix |
|---|---|---|
| 1 | Read a hand-exported `*.xlsx - Sheet1.csv` | Read the `.xlsx` directly; auto-locate the file |
| 2 | `extract_residential_prices` returned a **tuple** in the NaN branch but a `Series` otherwise -> inconsistent `.apply` shape | Always return a `pd.Series` |
| 3 | `Residential.*?Rent` regex can bleed into the Office/Shop section | Isolate the Residential **segment** (split on `||`) first |
| 4 | `count_amenities` counted blanks / duplicates / `N/A` fragments | Strip, drop blanks & `N/A`, de-duplicate case-insensitively |
| 5 | Dropped the original price column (loses provenance) | **Keep** it; add parsed columns alongside |
| 6 | No midpoints, no quality report | Add `res_avg_buy`/`res_avg_rent` + a parse/missingness report |
| 7 | `PINCODE` read as int (drops leading zeros, mixes dtypes) | Read as `string` |
| 8 | Repeated `inplace=True` mutation | Pure reassignments in small functions |

In [1]:
import re
from pathlib import Path
import numpy as np
import pandas as pd

# Locate the workbook whether the notebook runs from notebooks/ or the repo root
def find_file(name):
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / name).exists():
            return base / name
    raise FileNotFoundError(name)

XLSX = find_file("magicbricks_localities.xlsx")
ROOT = XLSX.parent
print("Reading:", XLSX)

Reading: C:\Users\singh\Desktop\GOATLife\magicbricks_localities.xlsx


In [2]:
PRICE_COL = "price range(for residential, office space, shop)"
AMENITY_COLS = ["Educational Institute", "Hospital", "Shopping Centre",
                "Transportation Hub", "Commercial Hub", "Tourist Spot", "Nearby Localities"]
TEXT_COLS = ["Physical infrastructure", "Locality introduction and neighbourhood",
             "Social & retail infra", "Nearby employment hubs"]

# 1) Load — pincode as string so leading zeros / mixed types are preserved
df = pd.read_excel(XLSX, dtype={"PINCODE": "string"})
print("Original shape:", df.shape)

# 2) Standardise missing values: 'N/A', 'NA', blank -> NaN (whole-cell match)
df = df.replace(r"^\s*(N/?A)?\s*$", np.nan, regex=True)

Original shape: (600, 16)


In [3]:
# 3) Parse the dense price string -> Residential buy/rent min & max (+ midpoints)
_NUM = r"Rs\.?\s*([\d,]+)\s*-\s*Rs\.?\s*([\d,]+)"

def _to_float(s):
    return float(s.replace(",", ""))

def extract_residential_prices(price_string):
    """Return res_min/max_buy and res_min/max_rent (Rs/sqft). Always a Series."""
    cols = ["res_min_buy", "res_max_buy", "res_min_rent", "res_max_rent"]
    vals = {c: np.nan for c in cols}
    if pd.isna(price_string):
        return pd.Series(vals)
    # Isolate the Residential segment so Office/Shop figures never leak in
    segment = next((s for s in re.split(r"\|\|", str(price_string)) if "Residential" in s), "")
    buy = re.search(r"Buy\s*" + _NUM, segment)
    if buy:
        vals["res_min_buy"], vals["res_max_buy"] = _to_float(buy.group(1)), _to_float(buy.group(2))
    rent = re.search(r"Rent\s*" + _NUM, segment)
    if rent:
        vals["res_min_rent"], vals["res_max_rent"] = _to_float(rent.group(1)), _to_float(rent.group(2))
    return pd.Series(vals)

prices = df[PRICE_COL].apply(extract_residential_prices)
df = pd.concat([df, prices], axis=1)
df["res_avg_buy"] = df[["res_min_buy", "res_max_buy"]].mean(axis=1)
df["res_avg_rent"] = df[["res_min_rent", "res_max_rent"]].mean(axis=1)

In [4]:
# 4) Amenity lists -> de-duplicated counts
def count_amenities(value):
    if pd.isna(value):
        return 0
    items = [p.strip() for p in str(value).split(",")]
    items = [p for p in items if p and p.upper() != "N/A"]
    return len({p.lower() for p in items})  # case-insensitive de-dup

for col in AMENITY_COLS:
    if col in df.columns:
        df["num_" + col.lower().replace(" ", "_")] = df[col].apply(count_amenities)

In [5]:
# 5) Normalise the heavy free-text blocks (collapse whitespace / newlines)
def clean_text(value):
    if pd.isna(value):
        return np.nan
    text = re.sub(r"\s+", " ", str(value)).strip()
    return text or np.nan

for col in TEXT_COLS:
    if col in df.columns:
        df[col] = df[col].apply(clean_text)

In [6]:
# 6) Data-quality report
n = len(df)
buy_ok = df["res_min_buy"].notna().sum()
rent_ok = df["res_min_rent"].notna().sum()
print("Rows:", n)
print("Residential buy parsed :", buy_ok, "/", n, "(%.1f%%)" % (buy_ok / n * 100))
print("Residential rent parsed:", rent_ok, "/", n, "(%.1f%%)" % (rent_ok / n * 100))
print()
print("Amenity count summary (mean / max):")
num_cols = [c for c in df.columns if c.startswith("num_")]
print(df[num_cols].describe().loc[["mean", "max"]].round(2).to_string())

Rows: 600
Residential buy parsed : 351 / 600 (58.5%)
Residential rent parsed: 351 / 600 (58.5%)

Amenity count summary (mean / max):
      num_educational_institute  num_hospital  num_shopping_centre  num_transportation_hub  num_commercial_hub  num_tourist_spot  num_nearby_localities
mean                        2.4          0.89                 1.04                    0.95                0.66              0.19                   5.67
max                         6.0          6.00                 6.00                    6.00                6.00              6.00                   7.00


In [7]:
# 7) Preview + save the structured table (original price column kept for provenance)
preview = df[["AREA", "res_min_buy", "res_max_buy", "res_avg_buy",
              "num_hospital", "num_educational_institute"]].head()
print(preview.to_string(index=False))

out = ROOT / "structured_magicbricks_localities.csv"
df.to_csv(out, index=False)
print()
print("Saved", df.shape[0], "rows x", df.shape[1], "cols ->", out.name)

                                AREA  res_min_buy  res_max_buy  res_avg_buy  num_hospital  num_educational_institute
                 Sohna Road, Gurgaon       8700.0      15200.0      11950.0             0                          5
                  Sector 46, Gurgaon       9400.0      15300.0      12350.0             0                          5
                  Sector 57, Gurgaon       9700.0      15600.0      12650.0             0                          4
Block C Sushant Lok Phase 1, Gurgaon      10900.0      17500.0      14200.0             0                          0
           Golf Course Road, Gurgaon      19900.0      33200.0      26550.0             0                          1

Saved 600 rows x 29 cols -> structured_magicbricks_localities.csv


## Improvements summary
- Robust `.xlsx` loading with auto file-location and string pincodes.
- `extract_residential_prices` always returns a `Series` (no shape bug) and isolates the Residential
  segment, so Office/Shop prices can't leak in.
- De-duplicated, blank-safe amenity counts.
- Added price **midpoints** and a **data-quality report** (parse rates + amenity stats).
- Kept the original price column for provenance instead of dropping it.

Output: `structured_magicbricks_localities.csv` — ready for the GOAT-Fit pipeline.